In [2]:
# ── Hücre 1: Kurulumlar ve Sabit Tanımlamaları ─────────────────────────────────
!pip install scikit-multilearn -q
!pip install 'ultralytics>=8.3' -q

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time
import torch
import torch.nn as nn

# ── Yollar (Paths) ─────────────────────────────────────────────────────────────
ROOT       = Path('/content/drive/MyDrive/Wrist Dataset')
LABELS_DIR = ROOT / 'folder_structure/yolov5/labels'
IMAGES_DIR = ROOT / 'images'
CSV_PATH   = ROOT / 'dataset.csv'

LOCAL_DIR  = Path('/content')
CACHE_PATH = ROOT / 'annotations_cache.csv'

# ── Konfigürasyon ──────────────────────────────────────────────────────────────
TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.10
TEST_RATIO    = 0.20
SEED          = 42
FORCE_REBUILD = False

CLASSES = ['boneanomaly', 'bonelesion', 'foreignbody_metal',
           'fracture', 'periostealreaction', 'pronatorsign', 'softtissue']

AGE_BINS   = [0, 5, 9, 13, 99]
AGE_LABELS = ['0-4', '5-8', '9-12', '13-19']



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.0 MB/s eta 0:00:00
Mounted at /content/drive


In [3]:
# ── Hücre 2: Proje İş Mantığı (Business Logic) ─────────────────────────────────

def remap_class(original_id):
    CLASS_REMAP = {0:0, 1:1, 2:2, 3:3, 4:2, 5:4, 6:5, 7:6, 8:None}
    if original_id not in CLASS_REMAP:
        raise KeyError(f'Bilinmeyen sınıf ID: {original_id}')
    return CLASS_REMAP[original_id]

def get_age_group(age):
    res = pd.cut([age], bins=AGE_BINS, labels=AGE_LABELS, right=False)
    return str(res[0])

def check_patient_leakage(train_set, val_set, test_set):
    train_set, val_set, test_set = set(train_set), set(val_set), set(test_set)
    if train_set & val_set: raise ValueError("SIZINTI: Train ve Val ortak!")
    if train_set & test_set: raise ValueError("SIZINTI: Train ve Test ortak!")
    if val_set & test_set: raise ValueError("SIZINTI: Val ve Test ortak!")
    return True

def flip_bounding_box_horizontal(x_center, image_width):
    if x_center < 0 or x_center > image_width:
        raise ValueError("Koordinat görüntü sınırları dışında olamaz!")
    return image_width - x_center

class DummyAttentionLayer(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.attention_weights = nn.Parameter(torch.ones(1, channels, 1, 1))
    def forward(self, x):
        return x * self.attention_weights



In [4]:
# ── Hücre 3: YOLOv11m Custom Architecture (CBAM) ───────────────────────────────
custom_yaml_path = '/content/yolov11m-cbam.yaml'

yaml_content = """
nc: 7
scales:
  m: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 1, CBAM, [1024]] # [US-3] ATTENTION KATMANI

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]
  - [[16, 19, 22], 1, Detect, [nc]]
"""

with open(custom_yaml_path, 'w') as f:
    f.write(yaml_content)

In [5]:
# ── Hücre 4: Bütünleşik Unit Testler (Görev US-1, US-2, US-3) ──────────────────
import unittest

class TestWristTraumaProject(unittest.TestCase):

    def test_flip_horizontal(self):
        self.assertEqual(flip_bounding_box_horizontal(100, 640), 540)
        with self.assertRaises(ValueError):
            flip_bounding_box_horizontal(700, 640)

    def test_remap_class(self):
        self.assertEqual(remap_class(4), 2)
        self.assertIsNone(remap_class(8))
        with self.assertRaises(KeyError):
            remap_class(99)

    def test_patient_leakage(self):
        self.assertTrue(check_patient_leakage(['P1', 'P2'], ['P3'], ['P4']))
        with self.assertRaises(ValueError):
            check_patient_leakage(['P1', 'P2'], ['P2', 'P3'], ['P4'])

    def test_age_grouping(self):
        self.assertEqual(get_age_group(12), '9-12')

    def test_attention_output_shape(self):
        dummy_input = torch.randn(4, 64, 128, 128)
        attention_layer = DummyAttentionLayer(channels=64)
        output = attention_layer(dummy_input)
        self.assertEqual(output.shape, dummy_input.shape)

unittest.main(argv=['first-arg-is-ignored'], exit=False)

.....
----------------------------------------------------------------------
Ran 5 tests in 0.106s

OK


In [ ]:
# ── Hücre 5: Veri Okuma, Ön İşleme ve Stratified Split ─────────────────────────
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
from skmultilearn.model_selection import IterativeStratification

t0 = time.time()

def _read_label(p):
    class_ids, annotations = set(), []
    for line in p.read_text().split('\n'):
        tok = line.split()
        if not tok: continue
        orig = int(tok[0])
        nid = remap_class(orig) # US-2 Test edilen fonksiyon
        if nid is not None:
            class_ids.add(nid)
            if len(tok) >= 5:
                annotations.append((nid, tok[1], tok[2], tok[3], tok[4]))
    return p.stem, class_ids, annotations

label_files = sorted(LABELS_DIR.glob('*.txt'))
print(f'Etiketler taranıyor ({len(label_files)} dosya)...', flush=True)

with ThreadPoolExecutor(max_workers=8) as ex:
    triples = list(ex.map(_read_label, label_files))

img_records = {k: {'new_classes': v, 'is_empty': len(v) == 0} for k, v, _ in triples}

pid_of  = {s: s.split('_')[0] for s in img_records}
pid_cls = defaultdict(set)
pid_ann = defaultdict(bool)
for stem, rec in img_records.items():
    pid = pid_of[stem]
    pid_cls[pid].update(rec['new_classes'])
    pid_ann[pid] = pid_ann[pid] or (not rec['is_empty'])

valid_pids = {p for p, v in pid_ann.items() if v}
df_m = pd.read_csv(CSV_PATH, dtype={'patient_id': str})
df_m['patient_id'] = df_m['patient_id'].str.zfill(4)
df_m = df_m[df_m['patient_id'].isin(valid_pids)].copy()

agg = (df_m.groupby('patient_id', sort=False)
           .agg(gender  = ('gender', 'first'),
                min_age = ('age', 'min'),
                lat_set = ('laterality', lambda x: frozenset(x.dropna())))
           .reset_index())

agg['age_group']  = pd.cut(agg['min_age'], bins=AGE_BINS, labels=AGE_LABELS, right=False).astype(str)
agg['laterality'] = agg['lat_set'].apply(lambda s: 'both' if len(s) > 1 else f'{list(s)[0]}_only')

for cid, cname in enumerate(CLASSES):
    agg[cname] = agg['patient_id'].apply(lambda pid, c=cid: int(c in pid_cls.get(pid, set())))

patient_df = agg.set_index('patient_id')[['gender', 'age_group', 'laterality'] + CLASSES]

enc  = pd.get_dummies(patient_df, columns=['gender', 'age_group', 'laterality'], dtype=int)
pids = enc.index.to_numpy()
lbl  = enc.values.astype(float)
idx  = np.random.default_rng(SEED).permutation(len(pids))
pids, lbl = pids[idx], lbl[idx]

s1 = IterativeStratification(n_splits=2, order=1, sample_distribution_per_fold=[1 - TRAIN_RATIO, TRAIN_RATIO])
f0, f1 = next(s1.split(sp.lil_matrix(lbl), sp.lil_matrix(lbl)))
tr_i, tp_i = (f1, f0) if len(f1) > len(f0) else (f0, f1)
train_pids  = pids[tr_i]
tp, tp_lbl  = pids[tp_i], lbl[tp_i]

vf = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
s2 = IterativeStratification(n_splits=2, order=1, sample_distribution_per_fold=[1 - vf, vf])
g0, g1 = next(s2.split(sp.lil_matrix(tp_lbl), sp.lil_matrix(tp_lbl)))
v_i, te_i = (g1, g0) if len(g1) < len(g0) else (g0, g1)
val_pids   = tp[v_i]
test_pids  = tp[te_i]

pid_split = {'train': set(train_pids), 'val': set(val_pids), 'test': set(test_pids)}

check_patient_leakage(pid_split['train'], pid_split['val'], pid_split['test'])
print('✓ Hasta sızıntısı (Leakage) testi başarılı. Sızıntı yok!')

for sn, pid_set in pid_split.items():
    rows = []
    for stem, rec in img_records.items():
        if rec['is_empty']: continue
        pid = pid_of[stem]
        if pid not in pid_set: continue
        p = patient_df.loc[pid]
        rows.append({'image_path': str(IMAGES_DIR / f'{stem}.png'), 'image_stem': stem, 'patient_id': pid,
                     'gender': p['gender'], 'age_group': p['age_group'], 'laterality': p['laterality'],
                     **{c: int(p[c]) for c in CLASSES}})
    pd.DataFrame(rows).sort_values('image_path').reset_index(drop=True).to_csv(LOCAL_DIR / f'{sn}_split{SEED}.csv', index=False)

Etiketler taranıyor (20327 dosya)...
✓ Hasta sızıntısı (Leakage) testi başarılı. Sızıntı yok!
✓ Hücre 5 Tamamlandı: Veri başarıyla bölündü. Süre: 432.8sn


In [ ]:
# ── Hücre 6: YOLO Formatını Hazırlama ──────────────────────────────────────────
import shutil

LOCAL_IMGS = Path('/content/images_local')
LOCAL_IMGS.mkdir(exist_ok=True)

all_stems = set()
for sn in ['train', 'val', 'test']:
    all_stems.update(pd.read_csv(LOCAL_DIR / f'{sn}_split{SEED}.csv')['image_stem'])

to_copy = [s for s in all_stems if not (LOCAL_IMGS / f'{s}.png').exists()]
if to_copy:
    print(f'{len(to_copy)} resim kopyalanıyor (Bu işlem biraz sürebilir)...')
    def _copy_img(stem): shutil.copy2(IMAGES_DIR / f'{stem}.png', LOCAL_IMGS / f'{stem}.png')
    with ThreadPoolExecutor(max_workers=32) as ex: list(ex.map(_copy_img, to_copy))

anno_map = {}
for stem, _, annotations in triples:
    if annotations:
        anno_map[stem] = [f"{c} {x} {y} {w} {h}" for c, x, y, w, h in annotations]

YOLO = Path(f'/content/yolo_data_{SEED}')
for split in ['train', 'val', 'test']:
    (YOLO / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO / 'labels' / split).mkdir(parents=True, exist_ok=True)

def _prep(args):
    stem, sn = args
    img_dst = YOLO / 'images' / sn / f'{stem}.png'
    if not img_dst.exists(): img_dst.symlink_to(LOCAL_IMGS / f'{stem}.png')
    lbl_dst = YOLO / 'labels' / sn / f'{stem}.txt'
    if not lbl_dst.exists(): lbl_dst.write_text('\n'.join(anno_map.get(stem, [])))

tasks = [(stem, sn) for sn in ['train', 'val', 'test'] for stem in pd.read_csv(LOCAL_DIR / f'{sn}_split{SEED}.csv')['image_stem']]
with ThreadPoolExecutor(max_workers=8) as ex: list(ex.map(_prep, tasks))

YOLO_YAML = YOLO / 'dataset.yaml'
YOLO_YAML.write_text(
    f'path: {YOLO}\n'
    f'train: images/train\n'
    f'val:   images/val\n'
    f'test:  images/test\n\n'
    f'nc: {len(CLASSES)}\n'
    f'names: {CLASSES}\n'
)

In [ ]:
# ── Hücre 7: Özel YOLOv11m-CBAM Model Eğitimi ve Test ─────────────────────────
from ultralytics import YOLO
from IPython.display import Image, display

print("YOLOv11m (Attention Katmanlı) başlatılıyor...")

model = YOLO('/content/yolov11m-cbam.yaml')

results = model.train(
    data      = str(YOLO_YAML),
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,
    patience  = 20,
    optimizer = 'AdamW',
    lr0       = 1e-3,
    cos_lr    = True,
    cache     = 'ram',
    fliplr    = 0.0,
    flipud    = 0.0,
    degrees   = 5,
    translate = 0.05,
    scale     = 0.3,
    device    = 0,
    project   = '/content/runs',
    name      = f'yolov11m_cbam_seed{SEED}',
    exist_ok  = True,
    save      = True,
    plots     = True,
)

# Test-set evaluation
run_dir = Path(f'/content/runs/yolov11m_cbam_seed{SEED}')
best_pt = run_dir / 'weights/best.pt'

print(f'\nEn iyi ağırlıklar yükleniyor: {best_pt}')
test_model = YOLO(best_pt)

test_res = test_model.val(
    data      = str(YOLO_YAML),
    split     = 'test',
    device    = 0,
    plots     = True,
    save_json = True,
)